# https://velog.io/@hyungraelee/Titanic-with-EDA

In [ ]:
!pip install missingno

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import missingno as msno

plt.rc('font', family='Malgun Gothic')
plt.rc('axes', unicode_minus=False)

import warnings
warnings.filterwarnings(action='ignore') 

### 1. 데이터 수집
from google.colab import drive
drive.mount('/content/drive')

df_train = pd.read_csv('/content/drive/My Drive/Colab Notebooks/titanic/train.csv')

In [ ]:
# 분석 목적
# 생존 여부에 여부에 영향을 미친 주요 요인에 대한 체계적 분석

Column	Definition
PassengerId	승객 번호
Survived	생존 여부
Pclass	Ticket class (1:Upper, 2:Middle, 3:Lower)
Name	승객 이름
Sex	성
Age	나이
SibSp	배에 같이 탄 형제 및 배우자 수
Parch	배에 같이 탄 부모 및 아이들 수
Ticket	티켓 번호
Fare	지불 요금
Cabin	선실
Embarked	승선한 항구(C:Cherbourg, Q:Queenstown, S=Southampton)

In [ ]:
# 2. 전처리
#    i) 구조진단
df_train.info()
df_train.head(100)
df_train.shape  #(891, 12)

In [ ]:
#    ii) 정제 (결측치, 이상치)
df_train.isnull().sum()
msno.matrix(df_train) 

df_train["Age"]=df_train["Age"].fillna(df_train["Age"].median())

df_train["Embarked"]=df["Embarked"].fillna(
    df_train["Embarked"].mode()[0]
)

df_train.drop("Cabin",axis=1,inplace=True)

# 판단:

# Age → 중앙값 대체
# Cabin → 결측 과다 제거
# Embarked → 최빈값

# ① Age → 중앙값(median) 대체
# 판단 1: 결측이 너무 많지 않음

# 20%면 버리기 아까움.
# 891 → 714행

# 손실 큼

# 그래서 채우는 방향.

# 판단 2: 숫자형 변수

# Age는 연속형 데이터

# 10
# 22
# 35
# 80

# 숫자는 보통

# 평균
# 중앙값
# 예측모델

# 사용.
# 판단 3: 분포가 약간 치우침

# 확인

# df["Age"].skew()

# 예시

# 0.38

# → 약한 우측 왜도

# 평균보다 중앙값 안정.

# ② Cabin → 제거(drop)

# 결측

# 687 / 891
# ≈ 77%
# 판단 1: 결측 비율 매우 큼

# 기준(실무에서 많이 씀)

# 결측률	판단
# <10%	대체
# 10~30%	상황 판단
# 30~50%	신중
# >50%	제거 고려

# Cabin:

# 77%

# 너무 큼.

# 판단 2: 채워도 의미 없음
# 판단 3: 행 제거는 더 위험
# 데이터 붕괴.

# ③ Embarked → 최빈값(mode)
# 결측

# 2 / 891
# ≈ 0.2%

# 매우 적음.

# 값 확인

# df["Embarked"].value_counts()

# 예시

# S    644
# C    168
# Q     77

# S가 압도적.

# 판단 1: 범주형 데이터

# Embarked

# S
# C
# Q

# 평균 불가

# 중앙값 불가

# → 최빈값

# 결국 전처리 판단 공식은 이렇게 생각하면 돼요.

# 1. 결측률 얼마인가?
# ↓
# 2. 컬럼 타입이 무엇인가?
# ↓
# 3. 데이터 분포가 어떤가?
# ↓
# 4. 채우면 왜곡되는가?
# ↓
# 5. 제거하면 손실이 큰가?

# Titanic에 적용하면

# Age
# → 숫자형 + 20%
# → 중앙값

# Cabin
# → 77%
# → 컬럼 제거

# Embarked
# → 범주형 + 0.2%
# → 최빈값

In [ ]:
#    iii) 문자열 · 시계열
# | Name                       | Title |
# | -------------------------- | ----- |
# | Braund, Mr. Owen Harris    | Mr    |
# | Cumings, Mrs. John Bradley | Mrs   |
# | Heikkinen, Miss. Laina     | Miss  |

# Braund, Mr. Owen Harris
#          ↑
# 추출 결과
# Mr


# 왜 이렇게 하냐?

# 원래 Name:
# Braund, Mr. Owen Harris
# 정보가 너무 많음
# 하지만
# Mr
# Mrs
# Miss
# Master

df_train["Title"]=(
    df_train["Name"]
    .str.extract(" ([A-Za-z]+)\.")
)
df_train



In [ ]:
#    iv) 정리 · 변환

# 불필요 제거
# 이 부분은 단순히 컬럼 수 줄이기가 아니라,

# 예측에 도움이 되는 정보인가? 중복인가? 식별자인가?
# 를 판단해서 제거

df_train.drop(
["PassengerId","Name","Ticket"],
axis=1,
inplace=True
)

# 근거
# 승객번호 1번이라 생존했나?
# 승객번호 10번이라 사망했나?
# 번호와 생존은 관계 없음.

# 실무 에서는 아래를 제거
# 고유번호
# Primary Key
# ID
# UUID

# ② Name 제거

# 예시

# Braund, Mr. Owen Harris
# Cumings, Mrs. John Bradley

# 문제:
# 이름 종류가 너무 많음.
# 891명인데
# Name 고유값 ≈ 891개
# 거의 전부 다름.

# 모델 입장

# Braund → ?
# Owen → ?

# 의미 없음.

# 하지만 이름 안에 유용한 정보 있음.
# 하지만 이름 안에 유용한 정보 있음.

# 예:

# Mr
# Mrs
# Miss
# Master

# 그래서

# df["Title"] = (
#     df["Name"]
#     .str.extract(
#         " ([A-Za-z]+)\."
#     )
# )

# 만 추출.

# 그 후 제거.

# df.drop(
# "Name",
# axis=1
# )

# 판단 기준

# 고유값 너무 많음
# (High Cardinality)

# ③ Ticket 제거

# 예시

# A/5 21171
# PC 17599
# STON/O2 3101282

# 문제:

# 이게 생존과 직접 연결 안 됨.

# 확인
# df["Ticket"].nunique()
# 결과
# 681
# 891명 중
# 681개 종류
# 너무 많음.

# 컬럼	           제거 이유
# PassengerId	  단순 식별자
# Name	          고유값 과다
# Ticket	        종류 과다 + 의미 약함

df_train



In [ ]:
# 피처 엔지니어링
# 피처 엔지니어링(Feature Engineering) 은 데이터 분석/머신러닝에서 가장 중요한 단계 중 하나입니다.
# 한 줄 정의부터:
# 기존 데이터를 더 의미 있는 변수(Feature)로 변환하거나 새로 만드는 작업

# 원본 데이터
# ↓
# 모델이 이해하기 쉬운 정보로 가공

# 피처엔지니어링 이해하기

# | 종류    | 예시        |
# | ----- | --------- |
# | 결합    | 키+몸무게→BMI |
# | 분할    | 날짜→연·월·요일 |
# | 구간화   | 나이→연령대    |
# | 문자 추출 | 이름→호칭     |
# | 집계    | 구매금액→평균   |
# | 변환    | 로그 변환     |

# 전처리
# → 데이터를 깨끗하게 만드는 작업

# 피처 엔지니어링
# → 데이터를 똑똑하게 만드는 작업

# (1) FamilySize 생성

# 원본

# SibSp	Parch
# 1	0
# 2	1

# 질문

# 혼자인가?
# 가족이 많은가?

# 생성

# df["FamilySize"] = (
#     df["SibSp"]
#     +
#     df["Parch"]
#     +
#     1
# )

# 결과

# SibSp	Parch	FamilySize
# 1	0	2
# 2	1	4

# 의미

# 탑승 가족 수

#    v) 재구조화
df_train["FamilySize"] = (
    df_train["SibSp"]+
    df_train["Parch"]+1
)

# import pandas as pd

# df = pd.read_csv("train.csv")


# # 가족 수
# df["FamilySize"] = (
#     df["SibSp"]
#     +
#     df["Parch"]
#     +
#     1
# )


# # 혼자 여부
# df["IsAlone"] = (
#     df["FamilySize"] == 1
# ).astype(int)


# # 호칭
# df["Title"] = (
#     df["Name"]
#     .str.extract(
#         " ([A-Za-z]+)\."
#     )
# )


# # 연령대
# df["AgeGroup"] = pd.cut(
#     df["Age"],
#     [0,18,40,60,100]
# )


# # 요금 그룹
# df["FareLevel"] = pd.qcut(
#     df["Fare"],
#     4
# )


# # 티켓 그룹
# df["TicketGroup"] = (
#     df.groupby(
#         "Ticket"
#     )["Ticket"]
#     .transform(
#         "count"
#     )
# )

# print(
#     df[
#         [
#             "FamilySize",
#             "IsAlone",
#             "Title",
#             "AgeGroup",
#             "FareLevel",
#             "TicketGroup"
#         ]
#     ].head()
# )



In [ ]:
#    vi) 스케일링
# 1. 스케일링이란?
# 한 줄 정의:
# 서로 다른 크기의 숫자를 비슷한 기준으로 맞추는 것
# 머신러닝 딥러닝 에서 주로 수행
# 데이터 분석시에서 스케일링이 필요 없을수 있음

# | Age |  Fare |
# | --: | ----: |
# |  22 |  7.25 |
# |  38 | 71.28 |
# |  35 | 53.10 |

# 문제:

# Age   → 0~80
# Fare  → 0~500

# 크기 차이가 큼.
# | 모델      | 스케일링 |
# | ------- | ---- |
# | 로지스틱 회귀 | 추천   |
# | KNN     | 필수   |
# | SVM     | 필수   |
# | 신경망     | 추천   |
# | 트리      | 불필요  |
# | 랜덤포레스트  | 불필요  |

In [ ]:
#    vii) 정보 조합 · 파생변수
# 피처 엔지니어링(Feature Engineering) 에서 아주 많이 쓰는 구간화(Binning)
df_train["AgeGroup"] = pd.cut(
    df_train["Age"],
    bins=[0,18,40,60,100]
)
# 또는 아래처럼
# df_train["AgeGroup"] = pd.cut(
#     df_train["Age"],
#     bins=[0,18,40,60,100],
#     labels=[
#         "Child",
#         "Adult",
#         "Middle",
#         "Old"
#     ]
# )
df_train



In [ ]:
#    viii) 그룹기반

#    ix) 범주형 인코딩 (원핫인코딩)
df_train = pd.get_dummies(
    df_train,
    drop_first=True
)
# 

In [ ]:
print("\n기초통계량")
print(df_train.describe())


# ① PassengerId
# 평균 : 446
# 최소 : 1
# 최대 : 891

# 해석:

# 승객 번호일 뿐

# 분석 가치 없음.

# → 제거 대상

# df.drop(
# "PassengerId",
# axis=1
# )

# ② Survived (생존)
# 평균 0.383

# 여기 중요합니다.

# 생존 컬럼은

# 0 = 사망
# 1 = 생존

# 이라 평균 자체가 생존 비율입니다.
# 해석:

# 전체 승객 중
# 약 38% 생존
# 약 62% 사망

# ③ Pclass (객실 등급)
# 평균 : 2.31
# 중앙값 : 3

# 객실:

# 1 → 상류층
# 2 → 중산층
# 3 → 일반석

# 해석:

# 평균이 2.3
# 중앙값이 3

# ↓

# 3등석 승객 많음.

# 즉

# 데이터가 하위 객실에 치우침

# ④ Age (나이)
# 평균 : 29.36
# 표준편차 : 13.0

# 최소 : 0.42
# Q1 : 22
# 중앙값 : 28
# Q3 : 35
# 최대 : 80

# 해석:

# 평균
# 평균 승객 나이
# ≈ 29세
# 중앙값
# 절반은
# 28세 이하
# Q1
# 25% 승객
# 22세 이하
# Q3
# 75% 승객
# 35세 이하
# 최대
# 80세 존재

# ↓

# 노인 존재

# 평균

# 29.36

# 중앙값

# 28

# 차이 작음

# ↓

# 큰 왜도 없음
# 약간 우측 치우침

# ⑤ SibSp (형제·배우자 수)
# 평균 : 0.52
# 최대 : 8

# 해석:

# 대부분

# 혼자
# 또는
# 1명 동반

# 많음.

# 하지만

# 최대 8명

# ↓

# 대가족 존재.

# 중앙값

# 0

# ↓

# 절반 이상 혼자 탑승

# ⑥ Parch (부모·자녀 수)
# 평균 : 0.38
# 중앙값 : 0
# 최대 : 6

# 해석:

# 대부분

# 부모·자녀 없이 탑승

# 하지만 일부 가족 승객 있음.

# ⑦ Fare (운임) ← 매우 중요
# 평균 : 32
# 중앙값 : 14

# 최대 : 512

# 이거 보자마자 보이는 것:

# 평균

# 32

# 중앙값

# 14

# 차이 큼.

# ↓

# 오른쪽 꼬리.

# ↓

# 고가 티켓 존재.

# 사분위수

# Q1 = 7
# Q3 = 31

# 해석:

# 50% 승객
# 7~31 요금

# 그런데

# 최대 512

# ↓

# 엄청 비싼 표 있음.

# ↓

# 평균 왜곡 가능.

# ↓

# 중앙값 고려.

### 기초 통계량 인사이트
```
1. 생존률 약 38%

2. 승객 대부분 3등석

3. 평균 나이 약 29세

4. 절반 이상 가족 없이 탑승

5. 운임(Fare)은 우측 편향 심함
→ 스케일링 고려

6. Age는 큰 이상치 없음
→ 중앙값 대체 적절
```
